# Tutorial: Transform Faces with MediaPipe & GStreamer

Welcome! This notebook is a beginner-friendly guide to using the **Mediapipe GStreamer Plugins**. Even if you have never used Docker or facial tracking before, this guide will help you transform images and videos in minutes.

### What are we going to do?
Facial transformation works in three main steps:
1.  **Detection**: The AI finds 478 specific points on the face (called "Landmarks").
2.  **Mapping**: We decide which points should move. For example, to make someone smile, we move the corners of the mouth upward.
3.  **Warping**: A mathematical algorithm (MLS) stretches the pixels of the image to follow the moving points, creating a realistic effect.

### The Tools We Use
- **Docker**: A tool that runs our software in a "controlled box" (called a Container). This means you don't have to install complex AI libraries on your computer; they are already inside our **Docker Image**.
- **Python API**: We provide a Python function (`process_media`) that handles everything for you automatically.

---

## 1. Getting Started with Docker

### Step A: Install Docker
Before running any code, you need **Docker Desktop** installed and running on your computer.
- [Download for Windows](https://docs.docker.com/desktop/install/windows-install/)
- [Download for Mac](https://docs.docker.com/desktop/install/mac-install/)
- [Download for Linux](https://docs.docker.com/desktop/install/linux-install/)

### Step B: Verify Docker
Run the cell below. If you see a version number, you are ready!

In [ ]:
!docker --version

---

## 2. Setup: Pull the Plugins & Download Models

Run this to download the pre-built plugins image.

In [ ]:
!docker pull ducksouplab/mp_plugins:latest
!docker tag ducksouplab/mp_plugins:latest mp_plugins:latest

### Download the MediaPipe "Brain"
Fetch the AI model file required for face tracking.

In [ ]:
import os
import sys

# Add the parent directory to the path so we can import our wrapper
sys.path.append("..")
from mozza_process import process_media

!cd .. && chmod +x download_face_landmarker_model.sh && ./download_face_landmarker_model.sh

---

## 3. Step 1: Detect Landmarks

Let's see what the AI sees! We will take an image from the `assets/` folder and draw the 478 landmarks on it.

In [ ]:
from IPython.display import Image, display, Video

root_dir = os.path.abspath("..")
input_img = os.path.join(root_dir, "assets", "test_image.jpg")
output_img = os.path.abspath("tutorial_landmarks.png")
model_path = os.path.join(root_dir, "face_landmarker.task")

# Use the Python function directly!
process_media(
    input_path=input_img,
    output_path=output_img,
    mode="landmarks",
    model_path=model_path
)

display(Image(filename=output_img))

---

## 4. Step 2: Make Someone Smile (CPU Mode)

Apply the smile deformation using the `process_media` function. 

**Tip:** If you don't have an NVIDIA GPU, always use `mode="cpu"`. It works on all systems.

In [ ]:
output_cpu = os.path.abspath("tutorial_smile_cpu.png")
deform_path = os.path.join(root_dir, "smile.dfm")

process_media(
    input_path=input_img,
    output_path=output_cpu,
    mode="cpu",
    deform=deform_path,
    alpha=2.0,
    warp_mode="per-group-roi",
    show_landmarks=False,
    model_path=model_path
)

display(Image(filename=output_cpu))

---

## 5. Step 3: Process a Video

The function works exactly the same for videos. Let's transform a short clip.

In [ ]:
input_video = os.path.join(root_dir, "assets", "dynamic_video.mp4")
output_video = os.path.abspath("tutorial_video_smile.mp4")

process_media(
    input_path=input_video,
    output_path=output_video,
    mode="cpu",
    deform=deform_path,
    alpha=1.5,
    warp_mode="per-group-roi",
    show_landmarks=False,
    model_path=model_path
)

display(Video(output_video, width=600, embed=True))

---

## 6. Advanced Customization

### A. The Parameters
- `alpha`: Intensity of the effect (1.0 = normal, 2.0 = double).
- `warp_mode`: Use `"per-group-roi"` for high-quality localized warping.
- `show_landmarks`: Set to `True` to see the dots, `False` for a realistic result.

### B. How the DFM file works
The `.dfm` file contains rules that tell landmark points where to move based on their neighbors. 

**Landmark Map:** Refer to the [MediaPipe Face Mesh Map](https://storage.googleapis.com/mediapipe-assets/documentation/visualization/face_mesh_full_landmark_connections.png) to find indices for custom rules.